In [ ]:
%load_ext autoreload
%autoreload 2

### Update with BA data

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name not in ("CLSBN26", "PepCL_Manuscript"):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

In [ ]:
# imports
from Data.mhc_pseudo_seqs import mhc_seq_dict
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.log_odds import *
from src.plotting import *

# misc
import warnings
warnings.filterwarnings("ignore")
device="cuda"

In [ ]:
# change gpu
# GPU_ID = 0
# GPU_ID = 1
GPU_ID = 2
# GPU_ID = 3

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")

Using GPU: NVIDIA H100 80GB HBM3 (ID: 2)


### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### MS/BA train and test sets

In [ ]:
# ms train and test sets
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

In [ ]:
# ba train and test sets
ba_train_test_dir = DATA_DIR / "Training_Datasets" / "Processed" / "BA"
train_df_ba_for_updating=pd.read_csv(f"{ba_train_test_dir}/train_df_ba_for_updating.csv", index_col=0)
test_df_ba_for_updating=pd.read_csv(f"{ba_train_test_dir}/test_df_ba_for_updating.csv", index_col=0)

In [ ]:
# merge to create ms/ba train set

# assign domain
ms_train_data["domain_id"]=0
train_df_ba_for_updating["domain_id"]=1

ms_ba_train_data=pd.concat([ms_train_data, train_df_ba_for_updating])

# reset index
ms_ba_train_data.reset_index(inplace=True, drop=True)

# print stats
print_data_stats(ms_ba_train_data)

Total: 38,139,587
Pos: 782,643 | 2.05%
Neg: 37,356,944 | 97.95%
N alleles: 192


In [ ]:
# match allele and length distribution between domains in train
ms_ba_train_data["seq_len"]=ms_ba_train_data.seq.str.replace("$", "").apply(len)

ms_ba_train_data_matched_allele_length=filter_domain0_to_domain1_pairs(ms_ba_train_data, allele_col='allele', seq_len_col='seq_len', domain_col='domain_id')

# reset index
ms_ba_train_data_matched_allele_length.reset_index(inplace=True, drop=True)

# print stats
print_data_stats(ms_ba_train_data_matched_allele_length)

Total: 31,494,746
Pos: 652,352 | 2.07%
Neg: 30,842,394 | 97.93%
N alleles: 83


In [ ]:
# full processing for running
ms_ba_train_data_matched_allele_length=full_process_data(ms_ba_train_data_matched_allele_length, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_ba_train_data_matched_allele_length=create_seq_allele_column(ms_ba_train_data_matched_allele_length)

test_df_ba_for_updating=full_process_data(test_df_ba_for_updating, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
test_df_ba_for_updating=create_seq_allele_column(test_df_ba_for_updating)

ms_test_data=full_process_data(ms_test_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_test_data=create_seq_allele_column(ms_test_data)

### Load MHCPrime

In [ ]:
# setup tokenizer
tokenizer = AminoAcidTokenizer()

In [ ]:
# set base model params
base_model_params={
        'vocab_size': tokenizer.vocab_size,
        'embed_dim': 480,
        'fc_hidden_dims': [240, 120], 
        'fc_dropout': 0.1,
        'n_heads': 16,
        'max_peptide_len': 34,
        'max_mhc_len': 34,
        "n_peptide_layers": 2,
        "n_mhc_layers": 1,
        'n_concat_layers': 3,
        'add_pos_enc': True,
        'use_segment_embedder': True,
        'dim_feedforward_multiplier': 1,
        'debugging': False,
        'fc_output_dim': 1,
        'pooling': 'cls', # cls, mean, min, max, attn, for sequence ffnn head
        "ffnn_use_layernorm": False, 
        'use_feature_table': True,
        "feature_table": processed_feature_table,
        "sequence_embedder_feature_names": feature_names,
        'sequence_embedder_feature_mode': "concat",  # replace or concat
        'sequence_embedder_tokenizer': tokenizer,
        "peptide_start_pos": 1, 
        "mhc_start_pos": 35, 
        "pos_enc_type": "sincos", 
        "seed": 42,

        # peptide conformer
        "add_peptide_conformer": True, 
        "peptide_conformer_num_layers": 2,
        "peptide_conformer_n_heads": 16,
        "peptide_conformer_conv_kernel": 5,
        "peptide_conformer_dropout": 0.0,

        # mhc conformer
        "add_mhc_conformer": False, 
        "mhc_conformer_num_layers": 2,
        "mhc_conformer_n_heads": 16,
        "mhc_conformer_conv_kernel": 5,
        "mhc_conformer_dropout": 0.0,

        # fusion conformer parameters
        "add_fusion_conformer": True, 
        "fusion_conformer_num_layers": 1,
        "fusion_conformer_n_heads": 16,
        "fusion_conformer_conv_kernel": 7,
        "fusion_conformer_dropout": 0.0,
        "fusion_conformer_mask_bos": True,
        "ff_pre_post_swiglu": False,

        # legacy init
        "legacy_init_compat": True,
}

In [ ]:
%%time
# load base model (optionally, can load model_params directly)
# run_name = "MHCPrime_Base"
# run_dir = MODEL_CHECKPOINTS_DIR / run_name
# model_dir = run_dir / "Checkpoints"

# mhcprime=init_model(base_model_params, print_check=False, print_params=True)
# mhcprime.load_state_dict(torch.load(f"{model_dir}/model_final.pt"))
# mhcprime.eval()
# print("Model loaded from checkpoint...")
# clear_all_gpu_memory()

base_model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Base" / "Checkpoints"
ckpt_path = base_model_dir / "model_final.pt"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

clear_all_gpu_memory()

mhcprime = init_model(
    base_model_params,
    print_check=False,
    print_params=True,
)

state_dict = torch.load(
    ckpt_path,
    map_location="cpu",
)

mhcprime.load_state_dict(state_dict, strict=True)
mhcprime.to(device)
mhcprime.eval()

print("Model loaded from checkpoint...")
print(f"Checkpoint: {ckpt_path}")
print(f"Model device: {next(mhcprime.parameters()).device}")


[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
All GPU memory has been cleared.
Current GPU memory allocated: 0.10 GB
Current GPU memory reserved: 0.12 GB
CPU times: user 620 ms, sys: 388 ms, total: 1.01 s
Wall time: 1.02 s


### PepCL updating

#### Run PepCL

In [ ]:
# for classification based training
pepcl_base_params_cls=dict(
            tokenizer=tokenizer,
            device="cuda",
            steps = 200,
            output_dir = None,
            save_every_n_steps = 2000,
            seed = 42,
            num_workers = 16,
            collate_fn = collate_fn,
            num_ms_pos = 2000,
            num_nonms_pos = 2000,
            ms_neg_ratio = 1.0,
            nonms_neg_ratio = 1.0,
            auto_calibrate = True,
            calibration_steps = 1,
            ms_batch_size = 3072,
            nonms_batch_size = 3072,
            encoder_lr = 2e-4,
            decoder_lr = 2e-4,
            weight_decay = 0.01,
            warmup_ratio = 0.1,
            update_domain_embeddings = False,
            primary_task_id = 1,
            sequential_backprop = True,
            ms_loss_type = "logsmoothap",
            ms_loss_alpha = 10.0,
            ms_lambda = 1.0,
            nonms_loss_type = "logsmoothap",
            nonms_loss_alpha = 10.0,
            ft_lambda = 1.0,
            dr_lambda = 1e-5,
            dr_loss_type = "mse"
)

# for continuous value training
pepcl_base_params_con=dict(
            tokenizer=tokenizer,
            device="cuda",
            steps = 200,
            output_dir = None,
            save_every_n_steps = 2000,
            seed = 42,
            num_workers = 16,
            collate_fn = collate_fn,
            num_ms_pos = 2000,
            num_nonms_pos = 2000,
            ms_neg_ratio = 1.0,
            nonms_neg_ratio = 1.0,
            auto_calibrate = True,
            calibration_steps = 1,
            ms_batch_size = 3072,
            nonms_batch_size = 3072,
            encoder_lr = 2e-4,
            decoder_lr = 2e-4,
            weight_decay = 0.01,
            warmup_ratio = 0.1,
            update_domain_embeddings = False,
            primary_task_id = 1,
            sequential_backprop = True,
            ms_loss_type = "logsmoothap",
            ms_loss_alpha = 10.0,
            ms_lambda = 1.0,
            nonms_loss_type = "spearman",
            nonms_loss_alpha = 0.01,
            ft_lambda = 1.0,
            dr_lambda = 1e-5,
            dr_loss_type = "mse",
            continuous_target_col="ba_value",
            continuous_ft_sampling_mode="allele_aware",
)

In [ ]:
# init models

# classification models
ba_ft_cls = copy.deepcopy(mhcprime)
ba_pepcl_cls = copy.deepcopy(mhcprime)

# continuous value models
ba_ft_con = copy.deepcopy(mhcprime)
ba_pepcl_con = copy.deepcopy(mhcprime)
clear_all_gpu_memory()

All GPU memory has been cleared.
Current GPU memory allocated: 0.52 GB
Current GPU memory reserved: 0.56 GB


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=ba_ft_cls,
            train_df=ms_ba_train_data_matched_allele_length,
            use_replay = False,
            **pepcl_base_params_cls,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=83, A_eff=30.10, balance=0.363, target_pos/cycle=1536, q_pos median=13.0 [7, 97], reuse median=0.0033, reuse max=0.0693, neg_ratio=1.0
  FT: alleles=76, A_eff=22.25, balance=0.293, target_pos/cycle=1536, q_pos median=11.5 [1, 143], reuse median=0.0654, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [01:09<00:00,  2.87it/s, MS=0.000000, FT=0.010974, DR=0.000000]


CPU times: user 1min 33s, sys: 16.1 s, total: 1min 49s
Wall time: 1min 47s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=ba_pepcl_cls,
            train_df=ms_ba_train_data_matched_allele_length,
            use_replay = True,
            **pepcl_base_params_cls,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=83, A_eff=30.10, balance=0.363, target_pos/cycle=1536, q_pos median=13.0 [7, 97], reuse median=0.0033, reuse max=0.0693, neg_ratio=1.0
  FT: alleles=76, A_eff=22.25, balance=0.293, target_pos/cycle=1536, q_pos median=11.5 [1, 143], reuse median=0.0654, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [02:39<00:00,  1.26it/s, MS=0.011989, FT=0.014034, DR=0.006761]


CPU times: user 3min 1s, sys: 22.9 s, total: 3min 23s
Wall time: 3min 21s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=ba_ft_con,
            train_df=ms_ba_train_data_matched_allele_length,
            use_replay = False,
            **pepcl_base_params_con,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=83, A_eff=30.10, balance=0.363, target_pos/cycle=1536, q_pos median=13.0 [7, 97], reuse median=0.0033, reuse max=0.0693, neg_ratio=1.0
  FT: alleles=76, A_eff=22.25, balance=0.293, target_pos/cycle=1536, q_pos median=11.5 [1, 143], reuse median=0.0654, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [01:08<00:00,  2.92it/s, MS=0.000000, FT=0.104217, DR=0.000000]


CPU times: user 1min 34s, sys: 15.7 s, total: 1min 50s
Wall time: 1min 49s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=ba_pepcl_con,
            train_df=ms_ba_train_data_matched_allele_length,
            use_replay = True,
            **pepcl_base_params_con,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=83, A_eff=30.10, balance=0.363, target_pos/cycle=1536, q_pos median=13.0 [7, 97], reuse median=0.0033, reuse max=0.0693, neg_ratio=1.0
  FT: alleles=76, A_eff=22.25, balance=0.293, target_pos/cycle=1536, q_pos median=11.5 [1, 143], reuse median=0.0654, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [02:39<00:00,  1.26it/s, MS=0.029328, FT=0.104450, DR=0.019097]


CPU times: user 2min 59s, sys: 24.6 s, total: 3min 24s
Wall time: 3min 20s


In [ ]:
%%time
"""
Save models
"""
# save_models=True
save_models=False

models_to_save={

    # updated models
    "ba_ft_cls": ba_ft_cls,
    "ba_pepcl_cls": ba_pepcl_cls,
    "ba_ft_con": ba_ft_con,
    "ba_pepcl_con": ba_pepcl_con,

}

output_dir=MODEL_CHECKPOINTS_DIR / "BA_Models"
output_dir.mkdir(parents=True, exist_ok=True)

if save_models:
    for model_name, model in models_to_save.items():
        print(model_name)
        torch.save(model.state_dict(), f"{output_dir}/{model_name}.pt")

ba_ft_cls
ba_pepcl_cls
ba_ft_con
ba_pepcl_con
CPU times: user 358 ms, sys: 205 ms, total: 563 ms
Wall time: 556 ms


#### Inference

In [ ]:
# run mhcprime as well
base_and_ft_models={
    # updated models
    "ba_ft_cls": ba_ft_cls,
    "ba_pepcl_cls": ba_pepcl_cls,
    "ba_ft_con": ba_ft_con,
    "ba_pepcl_con": ba_pepcl_con,
}

In [ ]:
%%time
# ms_test_data
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ms_test_data, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ms_test_data[model_name]=output_scores.detach().cpu().tolist()

3214300it [01:19, 40192.39it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (874.3 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:09<00:00,  8.11it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:10<00:00,  8.04it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:08<00:00,  8.13it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:08<00:00,  8.14it/s]


CPU times: user 9min 59s, sys: 30.4 s, total: 10min 30s
Wall time: 10min 8s


In [ ]:
# add base model
base_model=copy.deepcopy(mhcprime)
base_and_ft_models["mhcprime"]=base_model

In [ ]:
%%time
# test_df_ba_for_updating
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(test_df_ba_for_updating, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    test_df_ba_for_updating[model_name]=output_scores.detach().cpu().tolist()

4221it [00:00, 45809.08it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (1.1 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 2/2 [00:02<00:00,  1.39s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 2/2 [00:03<00:00,  1.93s/it]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 2/2 [00:04<00:00,  2.10s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 2/2 [00:03<00:00,  1.75s/it]


CPU times: user 1.28 s, sys: 16 s, total: 17.3 s
Wall time: 19.8 s


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
%%time
"""
Save test sets
"""
save_dfs=False
# save_dfs=True

if save_dfs:
    # ms test set
    datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
    ms_test_data.to_csv(f"{datasets_dir}/ms_test_data.csv")

    # ba test set
    datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "BA"
    test_df_ba_for_updating.to_csv(f"{datasets_dir}/test_df_ba_for_updating.csv")

CPU times: user 1min 11s, sys: 2.91 s, total: 1min 14s
Wall time: 1min 14s
